# MAUDE NLP Classifier — ClinicalBERT Fine-Tuning (Kaggle)

This notebook fine-tunes `emilyalsentzer/Bio_ClinicalBERT` on MAUDE adverse event narratives.

**Before running:**
1. Enable GPU: Notebook Settings → Accelerator → **GPU T4 x1**
2. Add your HuggingFace token as a Kaggle Secret named `HF_TOKEN`:
   - Notebook → **Add-ons → Secrets → Add a new secret**
   - Name: `HF_TOKEN`, Value: your HF write token from https://huggingface.co/settings/tokens
3. (Optional) Add `OPENFDA_API_KEY` secret the same way for higher API rate limits

**After training completes:**
- The fine-tuned checkpoint is pushed to HF Hub at `mukundisb/maude-clinicalbert`
- Download `models/bert_model_ref.json` from the output and commit it to your repo

## 1. Verify GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU detected. Go to Notebook Settings → Accelerator → GPU T4 x1")
print(result.stdout)

## 2. Clone the repo

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/makymaverick/maude-nlp-classifier.git"
REPO_DIR = "/kaggle/working/maude-nlp-classifier"
BRANCH   = "master"

# If the dir exists but points to the wrong remote, delete and re-clone
if os.path.exists(REPO_DIR):
    result = subprocess.run(
        ["git", "-C", REPO_DIR, "remote", "get-url", "origin"],
        capture_output=True, text=True
    )
    current_remote = result.stdout.strip()
    if current_remote != REPO_URL:
        print(f"Wrong remote ({current_remote}). Deleting and re-cloning...")
        os.system(f"rm -rf {REPO_DIR}")
    else:
        print(f"Correct remote. Pulling latest from {BRANCH}...")
        os.system(f"git -C {REPO_DIR} fetch origin {BRANCH}")
        os.system(f"git -C {REPO_DIR} checkout {BRANCH}")
        os.system(f"git -C {REPO_DIR} reset --hard origin/{BRANCH}")

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
    os.system(f"git clone -b {BRANCH} {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
# Confirm we have the fix
out = subprocess.run(["grep", "-n", "full_weights", "src/model/bert_classifier.py"],
                     capture_output=True, text=True).stdout
print(f"Fix present: {'full_weights' in out}")
print(f"Working directory: {os.getcwd()}")

## 3. Install dependencies

Kaggle already has PyTorch with CUDA pre-installed — we only need to add the HuggingFace stack and MLflow.

In [ ]:
# Kaggle pre-installs: torch, numpy, pandas, scikit-learn, requests
# We only need to add the HuggingFace stack + MLflow
os.system(
    "pip install -q "
    "'transformers>=4.40.0' "
    "'accelerate>=0.29.0' "
    "'huggingface_hub>=0.22.0' "
    "'mlflow>=2.13.0' "
    "'apscheduler>=3.10.0'"
)
print("Dependencies installed.")

## 4. Configure secrets

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# Required: HuggingFace write token
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception:
    raise RuntimeError(
        "HF_TOKEN secret not found.\n"
        "Add it via: Notebook → Add-ons → Secrets → Add a new secret"
    )

# Optional: openFDA API key (higher rate limits)
try:
    os.environ["OPENFDA_API_KEY"] = secrets.get_secret("OPENFDA_API_KEY")
    print("OPENFDA_API_KEY loaded from Kaggle Secrets")
except Exception:
    print("OPENFDA_API_KEY not set — using unauthenticated rate limit (1,000 req/min). That's fine.")

# Keep MLflow local (no remote server needed)
os.environ["MLFLOW_TRACKING_URI"] = "mlruns"

## 5. Configure training parameters

Edit these before running the training cell.

In [ ]:
# ── Training configuration ─────────────────────────────────────────────────
RECORDS     = 10000   # MAUDE records to fetch. Start here; increase to 20k+ for best results.
EPOCHS      = 3       # Fine-tuning epochs. 3 is safe for first run.
LR          = 2e-5    # AdamW learning rate. Standard for BERT fine-tuning.
BATCH_SIZE  = 16      # Per-device batch size. T4 (16 GB) handles 16 comfortably.
MAX_LENGTH  = 256     # Token limit. 256 covers 95th percentile of MAUDE narratives.
HUB_REPO    = "mukundisb/maude-clinicalbert"  # Your HF Hub repo
DROP_UNK    = True    # Exclude UNKNOWN labels from training (recommended)
USE_CACHED  = False   # Set True to skip re-fetching if data/raw/maude_raw.csv exists
# ──────────────────────────────────────────────────────────────────────────

print(f"Training config:")
print(f"  Records:    {RECORDS:,}")
print(f"  Epochs:     {EPOCHS}")
print(f"  LR:         {LR}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max length: {MAX_LENGTH}")
print(f"  Hub repo:   {HUB_REPO}")
print(f"  Drop UNK:   {DROP_UNK}")

## 6. Run training

Expected runtime on Kaggle T4:
- Data fetch (~10k records): ~3–5 min
- Fine-tuning (3 epochs): ~20–30 min
- Cross-validation (3 folds × 2 epochs): ~25–35 min
- HF Hub push: ~2–3 min
- **Total: ~50–75 min**

In [ ]:
import sys

# Build CLI args from config above
argv = [
    "train_bert",
    "--records",    str(RECORDS),
    "--epochs",     str(EPOCHS),
    "--lr",         str(LR),
    "--batch-size", str(BATCH_SIZE),
    "--max-length", str(MAX_LENGTH),
    "--hub-repo",   HUB_REPO,
]
if DROP_UNK:
    argv.append("--drop-unknown")
if USE_CACHED:
    argv.append("--use-cached")

# Patch sys.argv so argparse in train_bert.py picks up our config
sys.argv = argv

# Run
from src.model.train_bert import main
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--records",      type=int,   default=5000)
parser.add_argument("--epochs",       type=int,   default=3)
parser.add_argument("--lr",           type=float, default=2e-5)
parser.add_argument("--batch-size",   type=int,   default=16)
parser.add_argument("--max-length",   type=int,   default=256)
parser.add_argument("--hub-repo",     type=str,   default=None)
parser.add_argument("--use-cached",   action="store_true")
parser.add_argument("--drop-unknown", action="store_true")
args = parser.parse_args()

main(args)

## 7. Verify outputs

In [ ]:
import json

ref_path = "models/bert_model_ref.json"
champion_path = "models/champion_metrics.json"

print("=== bert_model_ref.json ===")
with open(ref_path) as f:
    ref = json.load(f)
print(json.dumps(ref, indent=2))

print("\n=== champion_metrics.json ===")
with open(champion_path) as f:
    champion = json.load(f)
print(json.dumps(champion, indent=2))

if ref.get("hub_repo") and ref.get("hub_sha"):
    print(f"\nCheckpoint pushed to: https://huggingface.co/{ref['hub_repo']}")
else:
    print("\nWARNING: hub_sha is null — HF Hub push may have failed. Check logs above.")

## 8. Download files to commit back to your repo

Copy the output of the next cell, then on your local machine:

```bash
# After pasting the JSON content into your local models/bert_model_ref.json:
git add models/bert_model_ref.json models/champion_metrics.json
git commit -m "chore: update bert_model_ref.json after Kaggle training run"
git push
```

In [ ]:
print("\n=== COPY THIS → models/bert_model_ref.json ===")
with open("models/bert_model_ref.json") as f:
    print(f.read())

print("\n=== COPY THIS → models/champion_metrics.json ===")
with open("models/champion_metrics.json") as f:
    print(f.read())